# FREIGHT-MNet M0.5 Hybrid Baseline Notebook

This notebook implements the **M0.5 hybrid baseline** between the original M0 tabular baselines and the later MLP / D-CQHGT models.

The purpose is to test whether the strongest M0 signals are complementary:

1. **OD historical prior** captures FAF OD-pair temporal persistence from 2018--2022.
2. **LightGBM quantile regression** captures nonlinear demand / zone feature patterns.
3. **Hybrid models** combine both sources through validation-tuned ensembles and residual correction.

The main target remains annual FAF OD truck travel-time quantile prediction:

```text
truck_q25, truck_q50, truck_q75
```

The temporal split is fixed:

```text
train = 2018--2022
validation = 2023
test = 2024
```

No graph model is used in this notebook. This is still a tabular / historical hybrid baseline.

## 0. Modeling design

This notebook evaluates the following models.

| Model | Description | Why it matters |
|---|---|---|
| `global_train_median` | Global train median for q25/q50/q75 | Minimum sanity baseline. |
| `od_historical_prior` | Train-only OD historical median | Strong persistence baseline. |
| `lightgbm_direct` | LightGBM quantile model using current manifest features | Strong nonlinear tabular baseline. |
| `hist_lgbm_ensemble_global` | Validation-tuned convex blend of OD history and direct LightGBM | Tests complementarity with one shared blend weight. |
| `hist_lgbm_ensemble_per_quantile` | Validation-tuned blend with one weight per quantile | Allows different blending for q25/q50/q75. |
| `residual_lgbm_current_features` | LightGBM predicts residuals relative to OD historical prior | Main M0.5 residual correction model. |
| `residual_lgbm_prior_features` | Residual LightGBM using current features plus historical-prior features | Strongest train-only hybrid candidate. |
| `prior_feature_lgbm_direct` | Direct quantile LightGBM using current features plus historical-prior features | Tests whether direct learning with prior features beats residual learning. |

Important leakage rule:

- The historical prior is computed only from **train years 2018--2022**.
- For training rows, the residual target uses a **leave-one-training-row-out OD prior** so that a row's own label is not used to construct its baseline residual.
- Validation and test priors use the full train-year OD median.

## 1. Environment note

This notebook avoids the subprocess import preflight used in the previous notebook because some IDEs, especially PyCharm Scientific Mode, may inject a different user-site package path into subprocesses. That can make a subprocess see `numpy==2.x` even when the current Jupyter kernel is using `numpy<2`.

The code below validates the **current notebook kernel** only. If imports fail here, create and select the clean conda environment before continuing.

Recommended environment:

```bash
conda activate freight_mnet_m0
python -m ipykernel install --user --name freight_mnet_m0 --display-name "Python (freight_mnet_m0)"
```

In [1]:
# -----------------------------------------------------------------------------
# Current-kernel import preflight
# -----------------------------------------------------------------------------
# Keep these environment variables before importing pandas. They reduce the
# chance that pandas pulls incompatible optional acceleration libraries.
import os
os.environ.setdefault("PANDAS_USE_NUMEXPR", "0")
os.environ.setdefault("NUMEXPR_MAX_THREADS", "8")

import sys
import json
import time
import math
import shutil
import pickle
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Dict, List, Tuple, Optional, Any

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd

# Disable pandas' numexpr execution path after import as well.
try:
    pd.set_option("compute.use_numexpr", False)
except Exception:
    pass

from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor
import joblib

try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
    LIGHTGBM_AVAILABLE = True
except Exception as exc:
    LIGHTGBM_AVAILABLE = False
    LGBMRegressor = None
    print("LightGBM import failed. The notebook can use sklearn fallback models, but LightGBM is preferred.")
    print(repr(exc))

try:
    from IPython.display import display
except Exception:
    display = print

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("LightGBM available:", LIGHTGBM_AVAILABLE)

if int(np.__version__.split(".")[0]) >= 2:
    print("WARNING: numpy>=2 detected in the current kernel. If compiled extensions fail, use the clean numpy<2 environment.")

Python executable: E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Scripts\python.exe
Python version: 3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
numpy: 2.4.5
pandas: 2.3.3
LightGBM available: True


## 2. Experiment configuration

Edit only this cell for a normal run.

The default path matches the project layout used by the previous M0 notebook:

```text
E:\NetworkOptimization\pythonProject1\Data
```

The output directory will be:

```text
Data/10_experiments/m05_hybrid_baselines_v1_notebook/{scope}/
```

In [2]:
# -----------------------------------------------------------------------------
# Experiment configuration
# -----------------------------------------------------------------------------
@dataclass
class ExperimentConfig:
    # Project data root. Change this path if your Data directory is elsewhere.
    data_root: Path = Path(r"E:\NetworkOptimization\pythonProject1\Data")

    # Main paper scope is "east_plus_gulf". Use "all" for robustness later.
    scope: str = "east_plus_gulf"

    # Output run name.
    run_name: str = "m05_hybrid_baselines_v1_notebook"

    # Reproducibility.
    seed: int = 42
    overwrite: bool = True
    save_models: bool = True

    # Weighting. The weight column is never used as a predictive feature.
    use_sample_weight: bool = True
    weight_column: str = "obs_weight_sum"
    weight_clip_min: float = 0.05
    weight_clip_max: float = 20.0

    # Which M0.5 models to run.
    run_direct_lightgbm: bool = True
    run_ensembles: bool = True
    run_residual_current_features: bool = True
    run_residual_prior_features: bool = True
    run_prior_feature_direct_lightgbm: bool = True

    # Optional global validation residual calibration. Previous M0 results showed
    # this can hurt test performance, so it is disabled by default.
    run_global_val_median_calibration: bool = False

    # LightGBM hyperparameters. These are intentionally close to M0 settings.
    lgb_n_estimators: int = 2000
    lgb_learning_rate: float = 0.03
    lgb_num_leaves: int = 63
    lgb_min_child_samples: int = 50
    lgb_subsample: float = 0.90
    lgb_colsample_bytree: float = 0.90
    lgb_reg_lambda: float = 1.0
    lgb_early_stopping_rounds: int = 100
    n_jobs: int = -1

    # Fallback sklearn quantile boosting if LightGBM is unavailable.
    fallback_max_iter: int = 600
    fallback_learning_rate: float = 0.04
    fallback_max_leaf_nodes: int = 63

    # Convex ensemble weight grid.
    # Weight alpha means: alpha * OD_history + (1 - alpha) * LightGBM.
    ensemble_grid_step: float = 0.01

    # Output precision.
    float_output_precision: int = 6

config = ExperimentConfig()
np.random.seed(config.seed)
config

ExperimentConfig(data_root=WindowsPath('E:/NetworkOptimization/pythonProject1/Data'), scope='east_plus_gulf', run_name='m05_hybrid_baselines_v1_notebook', seed=42, overwrite=True, save_models=True, use_sample_weight=True, weight_column='obs_weight_sum', weight_clip_min=0.05, weight_clip_max=20.0, run_direct_lightgbm=True, run_ensembles=True, run_residual_current_features=True, run_residual_prior_features=True, run_prior_feature_direct_lightgbm=True, run_global_val_median_calibration=False, lgb_n_estimators=2000, lgb_learning_rate=0.03, lgb_num_leaves=63, lgb_min_child_samples=50, lgb_subsample=0.9, lgb_colsample_bytree=0.9, lgb_reg_lambda=1.0, lgb_early_stopping_rounds=100, n_jobs=-1, fallback_max_iter=600, fallback_learning_rate=0.04, fallback_max_leaf_nodes=63, ensemble_grid_step=0.01, float_output_precision=6)

In [3]:
# -----------------------------------------------------------------------------
# Path management
# -----------------------------------------------------------------------------
def normalize_scope(scope: str) -> str:
    """Normalize scope names used in filenames and output folders."""
    scope = str(scope).strip().lower().replace("-", "_")
    aliases = {
        "eastplusgulf": "east_plus_gulf",
        "east_plus_gulf": "east_plus_gulf",
        "all": "all",
    }
    return aliases.get(scope, scope)


def json_safe(obj: Any) -> Any:
    """Convert objects to JSON-safe values."""
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(x) for x in obj]
    return obj


scope = normalize_scope(config.scope)
model_ready_dir = config.data_root / "08_processed" / "model_ready"
metadata_dir = model_ready_dir / "_metadata"

supervised_path = model_ready_dir / f"freight_mnet_supervised_edges_2018_2024_{scope}.parquet"
manifest_path = metadata_dir / "freight_mnet_supervised_feature_manifest.json"

output_dir = config.data_root / "10_experiments" / config.run_name / scope
models_dir = output_dir / "models"

if output_dir.exists() and config.overwrite:
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

paths = {
    "data_root": config.data_root,
    "supervised_path": supervised_path,
    "manifest_path": manifest_path,
    "output_dir": output_dir,
    "models_dir": models_dir,
}

for name, path in paths.items():
    print(f"{name}: {path}")

if not supervised_path.exists():
    raise FileNotFoundError(f"Supervised table not found: {supervised_path}")
if not manifest_path.exists():
    raise FileNotFoundError(f"Feature manifest not found: {manifest_path}")

data_root: E:\NetworkOptimization\pythonProject1\Data
supervised_path: E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\freight_mnet_supervised_edges_2018_2024_east_plus_gulf.parquet
manifest_path: E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\_metadata\freight_mnet_supervised_feature_manifest.json
output_dir: E:\NetworkOptimization\pythonProject1\Data\10_experiments\m05_hybrid_baselines_v1_notebook\east_plus_gulf
models_dir: E:\NetworkOptimization\pythonProject1\Data\10_experiments\m05_hybrid_baselines_v1_notebook\east_plus_gulf\models


In [4]:
# -----------------------------------------------------------------------------
# Constants and leakage guards
# -----------------------------------------------------------------------------
LABEL_COLUMNS = ["truck_q25", "truck_q50", "truck_q75"]
TAUS = np.array([0.25, 0.50, 0.75], dtype=float)
TRAIN_YEARS = [2018, 2019, 2020, 2021, 2022]
VAL_YEAR = 2023
TEST_YEAR = 2024

ORIGIN_COL_CANDIDATES = ["faf_orig", "origin_faf", "orig_faf", "dms_orig"]
DEST_COL_CANDIDATES = ["faf_dest", "destination_faf", "dest_faf", "dms_dest"]
YEAR_COL_CANDIDATES = ["year", "Year", "YEAR"]
SPLIT_COL_CANDIDATES = ["split", "data_split"]

# These columns are useful for weighting or diagnostics but must not be used as
# predictive features in M0/M0.5 tabular models.
FORBIDDEN_EXACT_FEATURES = {
    "truck_q25", "truck_q50", "truck_q75",
    "truck_iqr", "truck_q75_q50_gap", "truck_q50_q25_gap", "truck_iqr_over_q50",
    "n_fmi_county_pairs", "obs_weight_sum", "obs_weight_mean", "obs_weight_max",
    "input_q25_weighted_mean", "input_q50_weighted_mean", "input_q75_weighted_mean",
    "input_q50_min", "input_q50_max",
}

FORBIDDEN_SUBSTRINGS = [
    "fmi", "input_q", "weighted_mean", "label", "target"
]

# Historical-prior feature names. They are computed train-only in this notebook.
HIST_PRED_COLUMNS = ["hist_truck_q25", "hist_truck_q50", "hist_truck_q75"]
HIST_EXTRA_COLUMNS = [
    "hist_truck_iqr",
    "hist_truck_q75_q50_gap",
    "hist_truck_q50_q25_gap",
    "hist_n_prior_years",
    "hist_has_prior",
]
HIST_FEATURE_COLUMNS = HIST_PRED_COLUMNS + HIST_EXTRA_COLUMNS


def infer_required_column(columns: List[str], candidates: List[str], label: str) -> str:
    """Infer a required column from candidate names."""
    for c in candidates:
        if c in columns:
            return c
    raise KeyError(f"Could not infer {label}. Tried candidates: {candidates}")


def validate_feature_columns(df: pd.DataFrame, feature_columns: List[str]) -> List[str]:
    """Validate feature columns and remove missing columns with a clear warning."""
    missing = [c for c in feature_columns if c not in df.columns]
    present = [c for c in feature_columns if c in df.columns]

    if missing:
        print(f"WARNING: {len(missing)} manifest features are missing from the supervised table.")
        print(missing[:20])

    forbidden_exact = [c for c in present if c in FORBIDDEN_EXACT_FEATURES]
    if forbidden_exact:
        raise ValueError(f"Leakage features detected in feature manifest: {forbidden_exact}")

    forbidden_substring = []
    for c in present:
        low = c.lower()
        if any(s in low for s in FORBIDDEN_SUBSTRINGS):
            # Allow normal columns that happen to contain safe words if needed.
            forbidden_substring.append(c)
    if forbidden_substring:
        raise ValueError(
            "Potential leakage-like feature names detected. Inspect before continuing: "
            + str(forbidden_substring)
        )

    return present

## 3. Data loading utilities

This section reads the model-ready supervised table and the feature manifest.

The feature manifest is trusted as the baseline feature source, but this notebook still applies a leakage guard before training any model.

In [5]:
# -----------------------------------------------------------------------------
# Data loading and split validation
# -----------------------------------------------------------------------------
def load_manifest(path: Path) -> List[str]:
    """Load model-ready feature columns from the feature manifest JSON."""
    with open(path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    if "feature_columns" not in manifest:
        raise KeyError("Feature manifest does not contain key 'feature_columns'.")
    return list(manifest["feature_columns"])


def load_supervised_table(path: Path) -> pd.DataFrame:
    """Load supervised edge table and normalize the row index."""
    df = pd.read_parquet(path)
    df = df.reset_index(drop=True)
    return df


def build_split_masks(df: pd.DataFrame, year_col: str) -> Dict[str, np.ndarray]:
    """Build train/validation/test masks from the fixed temporal split."""
    years = pd.to_numeric(df[year_col], errors="coerce").astype("Int64")
    masks = {
        "train": years.isin(TRAIN_YEARS).to_numpy(dtype=bool),
        "val": (years == VAL_YEAR).to_numpy(dtype=bool),
        "test": (years == TEST_YEAR).to_numpy(dtype=bool),
    }
    return masks


def validate_dataset(df: pd.DataFrame, feature_columns: List[str], year_col: str, masks: Dict[str, np.ndarray]) -> None:
    """Run basic dataset validity checks before modeling."""
    missing_labels = [c for c in LABEL_COLUMNS if c not in df.columns]
    if missing_labels:
        raise KeyError(f"Missing label columns: {missing_labels}")

    for split_name, mask in masks.items():
        if int(mask.sum()) == 0:
            raise ValueError(f"Split {split_name!r} has zero rows.")

    y = df[LABEL_COLUMNS].to_numpy(dtype=float)
    crossing_rate = float(np.mean((y[:, 0] > y[:, 1]) | (y[:, 1] > y[:, 2])))
    if crossing_rate > 0:
        print(f"WARNING: label quantile crossing rate = {crossing_rate:.6f}")
    else:
        print("Label monotonicity check passed: q25 <= q50 <= q75 for all rows.")

    print("Dataset shape:", df.shape)
    print("Feature columns:", len(feature_columns))
    print("Split counts:", {k: int(v.sum()) for k, v in masks.items()})
    print("Year counts:")
    print(df[year_col].value_counts().sort_index())

## 4. Metrics and prediction post-processing

All models are evaluated with the same metric function.

Primary metrics:

- `mae_q50`
- `mae_q75`
- `pinball_mean`
- `iqr_mae`
- `stress_top10_mae_q75`
- `sparse_bottom25_mae_q75`

Predicted quantiles are post-processed with nonnegative clipping and sorting for fair comparison to M0 baselines. Raw crossing rates are still reported.

In [6]:
# -----------------------------------------------------------------------------
# Metrics and post-processing
# -----------------------------------------------------------------------------
def clip_sort_quantiles(pred: np.ndarray) -> np.ndarray:
    """Clip predictions to nonnegative values and enforce q25 <= q50 <= q75 by sorting."""
    pred = np.asarray(pred, dtype=float).copy()
    pred = np.where(np.isfinite(pred), pred, 0.0)
    pred = np.maximum(pred, 0.0)
    pred = np.sort(pred, axis=1)
    return pred


def quantile_crossing_rate(pred: np.ndarray) -> float:
    """Return the fraction of rows with q25 > q50 or q50 > q75."""
    pred = np.asarray(pred, dtype=float)
    return float(np.mean((pred[:, 0] > pred[:, 1]) | (pred[:, 1] > pred[:, 2])))


def negative_prediction_rate(pred: np.ndarray) -> float:
    """Return the fraction of individual quantile predictions below zero."""
    pred = np.asarray(pred, dtype=float)
    return float(np.mean(pred < 0))


def pinball_matrix(y_true: np.ndarray, y_pred: np.ndarray, taus: np.ndarray = TAUS) -> np.ndarray:
    """Compute per-row, per-quantile pinball loss."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    diff = y_true - y_pred
    losses = np.maximum(taus * diff, (taus - 1.0) * diff)
    return losses


def weighted_average(values: np.ndarray, weights: Optional[np.ndarray] = None) -> float:
    """Compute a finite weighted average with fallback to an unweighted mean."""
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(values)
    if weights is None:
        return float(np.mean(values[finite]))
    weights = np.asarray(weights, dtype=float)
    finite = finite & np.isfinite(weights) & (weights > 0)
    if finite.sum() == 0:
        return float(np.mean(values[np.isfinite(values)]))
    return float(np.average(values[finite], weights=weights[finite]))


def compute_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    raw_pred: np.ndarray,
    part_df: pd.DataFrame,
    weights: Optional[np.ndarray],
    model: str,
    variant: str,
    split: str,
) -> Dict[str, Any]:
    """Compute the standard FREIGHT-MNet baseline metric set."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    raw_pred = np.asarray(raw_pred, dtype=float)

    abs_err = np.abs(y_pred - y_true)
    sq_err_q50 = (y_pred[:, 1] - y_true[:, 1]) ** 2
    pin = pinball_matrix(y_true, y_pred)
    pin_per_row = pin.mean(axis=1)

    true_iqr = y_true[:, 2] - y_true[:, 0]
    pred_iqr = y_pred[:, 2] - y_pred[:, 0]
    iqr_abs_err = np.abs(pred_iqr - true_iqr)

    out = {
        "model": model,
        "variant": variant,
        "split": split,
        "n_rows": int(len(y_true)),
        "mae_q25": float(abs_err[:, 0].mean()),
        "mae_q50": float(abs_err[:, 1].mean()),
        "mae_q75": float(abs_err[:, 2].mean()),
        "rmse_q50": float(np.sqrt(sq_err_q50.mean())),
        "pinball_q25": float(pin[:, 0].mean()),
        "pinball_q50": float(pin[:, 1].mean()),
        "pinball_q75": float(pin[:, 2].mean()),
        "pinball_mean": float(pin.mean()),
        "iqr_mae": float(iqr_abs_err.mean()),
        "bias_q25": float((y_pred[:, 0] - y_true[:, 0]).mean()),
        "bias_q50": float((y_pred[:, 1] - y_true[:, 1]).mean()),
        "bias_q75": float((y_pred[:, 2] - y_true[:, 2]).mean()),
        "pred_crossing_rate": quantile_crossing_rate(y_pred),
        "pred_negative_rate": negative_prediction_rate(y_pred),
        "raw_crossing_rate": quantile_crossing_rate(raw_pred),
        "raw_negative_rate": negative_prediction_rate(raw_pred),
        "q50_inside_pred_q25_q75_rate": float(np.mean((y_true[:, 1] >= y_pred[:, 0]) & (y_true[:, 1] <= y_pred[:, 2]))),
    }

    if weights is not None:
        out.update({
            "weighted_pinball_mean": weighted_average(pin_per_row, weights),
            "weighted_mae_q50": weighted_average(abs_err[:, 1], weights),
            "weighted_mae_q75": weighted_average(abs_err[:, 2], weights),
        })
    else:
        out.update({
            "weighted_pinball_mean": np.nan,
            "weighted_mae_q50": np.nan,
            "weighted_mae_q75": np.nan,
        })

    # Stress split by true q75 top decile within the current split.
    stress_q75_threshold = float(np.quantile(y_true[:, 2], 0.90))
    stress_q75_mask = y_true[:, 2] >= stress_q75_threshold
    out.update({
        "stress_top10_true_q75_threshold": stress_q75_threshold,
        "stress_top10_true_q75_n": int(stress_q75_mask.sum()),
        "stress_top10_mae_q75": float(abs_err[stress_q75_mask, 2].mean()),
        "stress_top10_pinball_q75": float(pin[stress_q75_mask, 2].mean()),
    })

    # Stress split by true IQR top decile.
    stress_iqr_threshold = float(np.quantile(true_iqr, 0.90))
    stress_iqr_mask = true_iqr >= stress_iqr_threshold
    out.update({
        "stress_top10_true_iqr_threshold": stress_iqr_threshold,
        "stress_top10_iqr_n": int(stress_iqr_mask.sum()),
        "stress_top10_iqr_mae": float(iqr_abs_err[stress_iqr_mask].mean()),
        "stress_top10_iqr_mae_q75": float(abs_err[stress_iqr_mask, 2].mean()),
    })

    # Sparse-label split by bottom quartile of n_fmi_county_pairs if available.
    if "n_fmi_county_pairs" in part_df.columns:
        support = pd.to_numeric(part_df["n_fmi_county_pairs"], errors="coerce").to_numpy(dtype=float)
        support_finite = support[np.isfinite(support)]
        if len(support_finite) > 0:
            sparse_threshold = float(np.nanquantile(support_finite, 0.25))
            sparse_mask = support <= sparse_threshold
            out.update({
                "sparse_bottom25_county_pairs_threshold": sparse_threshold,
                "sparse_bottom25_n": int(np.nansum(sparse_mask)),
                "sparse_bottom25_mae_q50": float(abs_err[sparse_mask, 1].mean()),
                "sparse_bottom25_mae_q75": float(abs_err[sparse_mask, 2].mean()),
                "sparse_bottom25_pinball_mean": float(pin[sparse_mask].mean()),
            })
        else:
            out.update({
                "sparse_bottom25_county_pairs_threshold": np.nan,
                "sparse_bottom25_n": 0,
                "sparse_bottom25_mae_q50": np.nan,
                "sparse_bottom25_mae_q75": np.nan,
                "sparse_bottom25_pinball_mean": np.nan,
            })
    else:
        out.update({
            "sparse_bottom25_county_pairs_threshold": np.nan,
            "sparse_bottom25_n": 0,
            "sparse_bottom25_mae_q50": np.nan,
            "sparse_bottom25_mae_q75": np.nan,
            "sparse_bottom25_pinball_mean": np.nan,
        })

    return out

## 5. Historical-prior construction

The train-only historical prior is the core of M0.5.

For validation and test rows:

```text
hist_qτ(o,d) = median of train-year truck_qτ for the same FAF OD pair
```

For train rows used to fit residual models:

```text
hist_qτ_train_row = median of other train rows for the same FAF OD pair
```

This leave-one-row-out design avoids constructing a train residual using the row's own label.

In [7]:
# -----------------------------------------------------------------------------
# Historical-prior features
# -----------------------------------------------------------------------------
def build_historical_prior_features(
    df: pd.DataFrame,
    train_mask: np.ndarray,
    orig_col: str,
    dest_col: str,
) -> Tuple[pd.DataFrame, Dict[str, float]]:
    """Build train-only historical-prior features.

    Validation and test rows receive the full train-period OD median.
    Train rows receive leave-one-training-row-out OD medians to avoid row-level
    leakage when fitting residual models.
    """
    df = df.reset_index(drop=True)
    train_df = df.loc[train_mask, [orig_col, dest_col] + LABEL_COLUMNS].copy()

    global_medians = train_df[LABEL_COLUMNS].median().to_dict()
    global_vector = np.array([global_medians[c] for c in LABEL_COLUMNS], dtype=float)

    grouped = train_df.groupby([orig_col, dest_col], dropna=False, sort=False)
    full_medians = grouped[LABEL_COLUMNS].median().rename(columns={
        "truck_q25": "hist_truck_q25",
        "truck_q50": "hist_truck_q50",
        "truck_q75": "hist_truck_q75",
    })
    full_counts = grouped.size().rename("hist_n_prior_years")

    hist_lookup = pd.concat([full_medians, full_counts], axis=1).reset_index()
    hist_features = df[[orig_col, dest_col]].merge(hist_lookup, on=[orig_col, dest_col], how="left")

    # Fill validation/test cold OD pairs with global medians.
    hist_features["hist_has_prior"] = hist_features["hist_n_prior_years"].notna().astype(float)
    hist_features["hist_n_prior_years"] = hist_features["hist_n_prior_years"].fillna(0.0)
    for label, hist_col in zip(LABEL_COLUMNS, HIST_PRED_COLUMNS):
        hist_features[hist_col] = hist_features[hist_col].fillna(global_medians[label])

    # Replace train-row priors with leave-one-training-row-out medians.
    train_indices = np.where(train_mask)[0]
    loo_values = np.zeros((len(train_indices), 3), dtype=float)
    loo_counts = np.zeros(len(train_indices), dtype=float)

    local_position = {idx: i for i, idx in enumerate(train_indices)}
    for _, group in train_df.groupby([orig_col, dest_col], dropna=False, sort=False):
        idxs = group.index.to_numpy()
        vals = group[LABEL_COLUMNS].to_numpy(dtype=float)
        n = len(idxs)
        for pos, idx in enumerate(idxs):
            local_i = local_position[idx]
            if n > 1:
                other_vals = np.delete(vals, pos, axis=0)
                loo_values[local_i, :] = np.median(other_vals, axis=0)
                loo_counts[local_i] = n - 1
            else:
                loo_values[local_i, :] = global_vector
                loo_counts[local_i] = 0

    hist_features.loc[train_indices, HIST_PRED_COLUMNS] = loo_values
    hist_features.loc[train_indices, "hist_n_prior_years"] = loo_counts
    hist_features.loc[train_indices, "hist_has_prior"] = (loo_counts > 0).astype(float)

    # Derived interval features.
    hist_features["hist_truck_iqr"] = hist_features["hist_truck_q75"] - hist_features["hist_truck_q25"]
    hist_features["hist_truck_q75_q50_gap"] = hist_features["hist_truck_q75"] - hist_features["hist_truck_q50"]
    hist_features["hist_truck_q50_q25_gap"] = hist_features["hist_truck_q50"] - hist_features["hist_truck_q25"]

    # Keep only the engineered historical features in a stable order.
    hist_features = hist_features[HIST_FEATURE_COLUMNS].astype(float)

    return hist_features, {k: float(v) for k, v in global_medians.items()}

## 6. Feature matrices and sample weights

The original 64 model-ready features are loaded from the feature manifest.

Historical-prior features are added only for hybrid models. These features are not read from the data file; they are constructed train-only in this notebook.

In [8]:
# -----------------------------------------------------------------------------
# Feature matrices and weights
# -----------------------------------------------------------------------------
def make_numeric_frame(df: pd.DataFrame, columns: List[str]) -> pd.DataFrame:
    """Convert selected columns to numeric values with NaN for invalid entries."""
    out = pd.DataFrame(index=df.index)
    for c in columns:
        out[c] = pd.to_numeric(df[c], errors="coerce")
    out = out.replace([np.inf, -np.inf], np.nan)
    return out


def fit_transform_features(
    df: pd.DataFrame,
    train_mask: np.ndarray,
    feature_columns: List[str],
) -> Tuple[np.ndarray, SimpleImputer]:
    """Fit a median imputer on train rows and transform all rows."""
    numeric = make_numeric_frame(df, feature_columns)
    imputer = SimpleImputer(strategy="median")
    imputer.fit(numeric.loc[train_mask, feature_columns])
    X = imputer.transform(numeric[feature_columns]).astype(np.float32)
    return X, imputer


def build_sample_weights(df: pd.DataFrame, train_mask: np.ndarray, config: ExperimentConfig) -> Optional[np.ndarray]:
    """Build normalized sample weights from obs_weight_sum if requested and available."""
    if not config.use_sample_weight:
        return None
    if config.weight_column not in df.columns:
        print(f"Weight column {config.weight_column!r} not found. Running unweighted models.")
        return None

    raw = pd.to_numeric(df[config.weight_column], errors="coerce").fillna(0.0).to_numpy(dtype=float)
    weights = np.log1p(np.maximum(raw, 0.0))
    train_mean = np.nanmean(weights[train_mask])
    if not np.isfinite(train_mean) or train_mean <= 0:
        print("Invalid train weight mean. Running unweighted models.")
        return None
    weights = weights / train_mean
    weights = np.clip(weights, config.weight_clip_min, config.weight_clip_max)
    return weights.astype(np.float32)


def matrix_by_split(X: np.ndarray, masks: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
    """Slice a full matrix into train/val/test matrices."""
    return {name: X[mask] for name, mask in masks.items()}


def labels_by_split(y: np.ndarray, masks: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
    """Slice labels into train/val/test arrays."""
    return {name: y[mask] for name, mask in masks.items()}


def weights_by_split(weights: Optional[np.ndarray], masks: Dict[str, np.ndarray]) -> Dict[str, Optional[np.ndarray]]:
    """Slice weights into train/val/test arrays."""
    if weights is None:
        return {name: None for name in masks}
    return {name: weights[mask] for name, mask in masks.items()}

## 7. Quantile model training functions

LightGBM is the preferred backend. If LightGBM is unavailable, the notebook falls back to sklearn's `HistGradientBoostingRegressor` with quantile loss.

Each quantile is trained as a separate model.

In [9]:
# -----------------------------------------------------------------------------
# Quantile boosting model training
# -----------------------------------------------------------------------------
def fit_quantile_boosting_models(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    X_all: np.ndarray,
    weights_train: Optional[np.ndarray],
    weights_val: Optional[np.ndarray],
    config: ExperimentConfig,
    model_name: str,
) -> Tuple[np.ndarray, List[Any]]:
    """Fit one quantile boosting model per target quantile and predict all rows."""
    models = []
    pred_all_columns = []

    for j, tau in enumerate(TAUS):
        print(f"Training {model_name}: quantile tau={tau:.2f}")

        if LIGHTGBM_AVAILABLE:
            model = LGBMRegressor(
                objective="quantile",
                alpha=float(tau),
                n_estimators=config.lgb_n_estimators,
                learning_rate=config.lgb_learning_rate,
                num_leaves=config.lgb_num_leaves,
                min_child_samples=config.lgb_min_child_samples,
                subsample=config.lgb_subsample,
                colsample_bytree=config.lgb_colsample_bytree,
                reg_lambda=config.lgb_reg_lambda,
                random_state=config.seed + j,
                n_jobs=config.n_jobs,
                verbosity=-1,
            )

            callbacks = []
            if config.lgb_early_stopping_rounds and config.lgb_early_stopping_rounds > 0:
                callbacks.append(lgb.early_stopping(config.lgb_early_stopping_rounds, verbose=False))
            callbacks.append(lgb.log_evaluation(period=0))

            fit_kwargs = {
                "X": X_train,
                "y": y_train[:, j],
                "eval_set": [(X_val, y_val[:, j])],
                "eval_metric": "quantile",
                "callbacks": callbacks,
            }
            if weights_train is not None:
                fit_kwargs["sample_weight"] = weights_train
            if weights_val is not None:
                fit_kwargs["eval_sample_weight"] = [weights_val]

            model.fit(**fit_kwargs)
            pred_col = model.predict(X_all)
        else:
            # Fallback path. This is slower and less tuned than LightGBM, but it
            # keeps the notebook runnable in minimal environments.
            model = HistGradientBoostingRegressor(
                loss="quantile",
                quantile=float(tau),
                max_iter=config.fallback_max_iter,
                learning_rate=config.fallback_learning_rate,
                max_leaf_nodes=config.fallback_max_leaf_nodes,
                random_state=config.seed + j,
            )
            model.fit(X_train, y_train[:, j], sample_weight=weights_train)
            pred_col = model.predict(X_all)

        models.append(model)
        pred_all_columns.append(pred_col.astype(float))

    pred_all = np.column_stack(pred_all_columns)
    return pred_all, models

## 8. Ensemble tuning functions

The ensemble is tuned on 2023 validation only.

For a global weight:

```text
q_hat = alpha * q_history + (1 - alpha) * q_lightgbm
```

For per-quantile weights:

```text
q_hat_τ = alpha_τ * q_history_τ + (1 - alpha_τ) * q_lightgbm_τ
```

In [10]:
# -----------------------------------------------------------------------------
# Validation-tuned convex ensembles
# -----------------------------------------------------------------------------
def pinball_mean_score(y_true: np.ndarray, y_pred: np.ndarray, weights: Optional[np.ndarray] = None) -> float:
    """Return weighted or unweighted mean pinball loss across all quantiles."""
    pin = pinball_matrix(y_true, y_pred).mean(axis=1)
    return weighted_average(pin, weights)


def tune_global_convex_weight(
    y_val: np.ndarray,
    pred_a_val: np.ndarray,
    pred_b_val: np.ndarray,
    weights_val: Optional[np.ndarray],
    grid_step: float,
) -> Tuple[float, pd.DataFrame]:
    """Tune one convex weight alpha on validation pinball mean."""
    rows = []
    grid = np.round(np.arange(0.0, 1.0 + 1e-12, grid_step), 6)
    for alpha in grid:
        pred = clip_sort_quantiles(alpha * pred_a_val + (1.0 - alpha) * pred_b_val)
        score = pinball_mean_score(y_val, pred, weights_val)
        rows.append({"alpha_history": float(alpha), "val_pinball_mean": float(score)})
    table = pd.DataFrame(rows).sort_values("val_pinball_mean", ascending=True).reset_index(drop=True)
    best_alpha = float(table.loc[0, "alpha_history"])
    return best_alpha, table


def tune_per_quantile_convex_weights(
    y_val: np.ndarray,
    pred_a_val: np.ndarray,
    pred_b_val: np.ndarray,
    weights_val: Optional[np.ndarray],
    grid_step: float,
) -> Tuple[np.ndarray, pd.DataFrame]:
    """Tune one convex weight per quantile using validation pinball loss."""
    best_alphas = []
    rows = []
    grid = np.round(np.arange(0.0, 1.0 + 1e-12, grid_step), 6)

    for j, tau in enumerate(TAUS):
        candidate_rows = []
        for alpha in grid:
            pred_col = alpha * pred_a_val[:, j] + (1.0 - alpha) * pred_b_val[:, j]
            diff = y_val[:, j] - pred_col
            loss = np.maximum(tau * diff, (tau - 1.0) * diff)
            score = weighted_average(loss, weights_val)
            candidate_rows.append({
                "quantile": LABEL_COLUMNS[j],
                "tau": float(tau),
                "alpha_history": float(alpha),
                "val_pinball": float(score),
            })
        q_table = pd.DataFrame(candidate_rows).sort_values("val_pinball", ascending=True).reset_index(drop=True)
        best_alphas.append(float(q_table.loc[0, "alpha_history"]))
        rows.extend(candidate_rows)

    return np.array(best_alphas, dtype=float), pd.DataFrame(rows)


def apply_per_quantile_blend(pred_a: np.ndarray, pred_b: np.ndarray, alphas: np.ndarray) -> np.ndarray:
    """Apply per-quantile convex weights."""
    return pred_a * alphas.reshape(1, -1) + pred_b * (1.0 - alphas.reshape(1, -1))

## 9. Output and evaluation registry

The registry below stores every model's metrics and predictions using a consistent format.

In [11]:
# -----------------------------------------------------------------------------
# Result registry and output helpers
# -----------------------------------------------------------------------------
metrics_rows: List[Dict[str, Any]] = []
prediction_tables: List[pd.DataFrame] = []
registered_predictions: Dict[str, np.ndarray] = {}
registered_raw_predictions: Dict[str, np.ndarray] = {}
model_notes: Dict[str, Any] = {}


def safe_write_table(df_out: pd.DataFrame, path_without_suffix: Path) -> Path:
    """Write a table as parquet if possible, otherwise fall back to CSV."""
    parquet_path = path_without_suffix.with_suffix(".parquet")
    csv_path = path_without_suffix.with_suffix(".csv")
    try:
        df_out.to_parquet(parquet_path, index=False)
        return parquet_path
    except Exception as exc:
        print(f"Parquet write failed for {parquet_path.name}; falling back to CSV. Reason: {repr(exc)}")
        df_out.to_csv(csv_path, index=False)
        return csv_path


def make_prediction_table(
    df: pd.DataFrame,
    masks: Dict[str, np.ndarray],
    pred_all: np.ndarray,
    raw_pred_all: np.ndarray,
    model: str,
    variant: str,
    orig_col: str,
    dest_col: str,
    year_col: str,
) -> pd.DataFrame:
    """Create a long prediction table for validation and test rows."""
    id_cols = [orig_col, dest_col, year_col]
    for optional in ["split", "faf_od", "scope", "n_fmi_county_pairs", "obs_weight_sum"]:
        if optional in df.columns and optional not in id_cols:
            id_cols.append(optional)

    keep_mask = masks["val"] | masks["test"]
    out = df.loc[keep_mask, id_cols + LABEL_COLUMNS].copy()
    out.insert(0, "variant", variant)
    out.insert(0, "model", model)
    out["pred_q25"] = pred_all[keep_mask, 0]
    out["pred_q50"] = pred_all[keep_mask, 1]
    out["pred_q75"] = pred_all[keep_mask, 2]
    out["raw_pred_q25"] = raw_pred_all[keep_mask, 0]
    out["raw_pred_q50"] = raw_pred_all[keep_mask, 1]
    out["raw_pred_q75"] = raw_pred_all[keep_mask, 2]
    out["resid_q25"] = out["pred_q25"] - out["truck_q25"]
    out["resid_q50"] = out["pred_q50"] - out["truck_q50"]
    out["resid_q75"] = out["pred_q75"] - out["truck_q75"]
    return out


def register_model_predictions(
    model: str,
    raw_pred_all: np.ndarray,
    df: pd.DataFrame,
    masks: Dict[str, np.ndarray],
    y_all: np.ndarray,
    weights_all: Optional[np.ndarray],
    orig_col: str,
    dest_col: str,
    year_col: str,
    variant: str = "postprocessed",
    extra_note: Optional[Dict[str, Any]] = None,
) -> np.ndarray:
    """Post-process, evaluate, and register a model's predictions."""
    raw_pred_all = np.asarray(raw_pred_all, dtype=float)
    pred_all = clip_sort_quantiles(raw_pred_all)

    key = f"{model}::{variant}"
    registered_raw_predictions[key] = raw_pred_all
    registered_predictions[key] = pred_all
    if extra_note:
        model_notes[key] = extra_note

    for split_name, mask in masks.items():
        part_df = df.loc[mask].copy()
        split_weights = None if weights_all is None else weights_all[mask]
        metrics_rows.append(compute_metrics(
            y_true=y_all[mask],
            y_pred=pred_all[mask],
            raw_pred=raw_pred_all[mask],
            part_df=part_df,
            weights=split_weights,
            model=model,
            variant=variant,
            split=split_name,
        ))

    prediction_tables.append(make_prediction_table(
        df=df,
        masks=masks,
        pred_all=pred_all,
        raw_pred_all=raw_pred_all,
        model=model,
        variant=variant,
        orig_col=orig_col,
        dest_col=dest_col,
        year_col=year_col,
    ))

    if config.run_global_val_median_calibration:
        val_mask = masks["val"]
        correction = np.median(y_all[val_mask] - pred_all[val_mask], axis=0)
        cal_raw = pred_all + correction.reshape(1, -1)
        cal_pred = clip_sort_quantiles(cal_raw)
        cal_key = f"{model}::val_median_calibrated"
        registered_raw_predictions[cal_key] = cal_raw
        registered_predictions[cal_key] = cal_pred
        model_notes[cal_key] = {"val_median_correction": correction.tolist()}
        for split_name, mask in masks.items():
            part_df = df.loc[mask].copy()
            split_weights = None if weights_all is None else weights_all[mask]
            row = compute_metrics(
                y_true=y_all[mask],
                y_pred=cal_pred[mask],
                raw_pred=cal_raw[mask],
                part_df=part_df,
                weights=split_weights,
                model=model,
                variant="val_median_calibrated",
                split=split_name,
            )
            row["cal_correction_q25"] = float(correction[0])
            row["cal_correction_q50"] = float(correction[1])
            row["cal_correction_q75"] = float(correction[2])
            metrics_rows.append(row)

    return pred_all

## 10. Load data

This cell loads the supervised table, infers ID columns, validates labels, and confirms the temporal split.

In [12]:
# -----------------------------------------------------------------------------
# Load supervised data and manifest
# -----------------------------------------------------------------------------
df = load_supervised_table(supervised_path)
manifest_feature_columns = load_manifest(manifest_path)

all_columns = list(df.columns)
orig_col = infer_required_column(all_columns, ORIGIN_COL_CANDIDATES, "origin FAF column")
dest_col = infer_required_column(all_columns, DEST_COL_CANDIDATES, "destination FAF column")
year_col = infer_required_column(all_columns, YEAR_COL_CANDIDATES, "year column")

feature_columns = validate_feature_columns(df, manifest_feature_columns)
masks = build_split_masks(df, year_col)
validate_dataset(df, feature_columns, year_col, masks)

print("Origin column:", orig_col)
print("Destination column:", dest_col)
print("Year column:", year_col)
print("First 10 features:", feature_columns[:10])

Label monotonicity check passed: q25 <= q50 <= q75 for all rows.
Dataset shape: (73972, 92)
Feature columns: 64
Split counts: {'train': 52773, 'val': 10625, 'test': 10574}
Year counts:
year
2018     9936
2019    10704
2020    10761
2021    10721
2022    10651
2023    10625
2024    10574
Name: count, dtype: int64
Origin column: faf_orig
Destination column: faf_dest
Year column: year
First 10 features: ['dest_east_plus_gulf_county_share', 'dest_east_plus_gulf_faf_flag_any_county', 'dest_max_county_centroid_lon', 'dest_min_county_centroid_lon', 'dest_n_counties', 'dest_n_east_plus_gulf_counties', 'has_multimodal_demand', 'has_rail_demand', 'has_truck_demand', 'log1p_tmiles_multimodal']


## 11. Build labels, weights, historical priors, and feature matrices

The matrices produced here are:

- `X_current`: original manifest features only.
- `X_current_plus_prior`: original features plus train-only historical-prior features.
- `prior_pred_all`: historical-prior q25/q50/q75 for every row.

In [13]:
# -----------------------------------------------------------------------------
# Build labels, sample weights, historical priors, and model matrices
# -----------------------------------------------------------------------------
y_all = df[LABEL_COLUMNS].to_numpy(dtype=float)
y_split = labels_by_split(y_all, masks)

weights_all = build_sample_weights(df, masks["train"], config)
w_split = weights_by_split(weights_all, masks)

hist_features, global_train_medians = build_historical_prior_features(
    df=df,
    train_mask=masks["train"],
    orig_col=orig_col,
    dest_col=dest_col,
)

# Attach historical features to a model dataframe. These features are generated
# by this notebook and are not original data-file columns.
df_model = pd.concat([df.reset_index(drop=True), hist_features.reset_index(drop=True)], axis=1)

prior_pred_all = hist_features[HIST_PRED_COLUMNS].to_numpy(dtype=float)
global_pred_all = np.tile(
    np.array([global_train_medians[c] for c in LABEL_COLUMNS], dtype=float).reshape(1, -1),
    (len(df), 1),
)

# Feature matrix with original manifest features only.
X_current, imputer_current = fit_transform_features(df_model, masks["train"], feature_columns)

# Feature matrix with original features plus train-only historical-prior features.
feature_columns_with_prior = feature_columns + HIST_FEATURE_COLUMNS
X_current_plus_prior, imputer_current_plus_prior = fit_transform_features(
    df_model,
    masks["train"],
    feature_columns_with_prior,
)

X_current_split = matrix_by_split(X_current, masks)
X_prior_split = matrix_by_split(X_current_plus_prior, masks)
prior_split = matrix_by_split(prior_pred_all, masks)

display(pd.DataFrame({
    "matrix": ["X_current", "X_current_plus_prior", "prior_pred_all"],
    "shape": [str(X_current.shape), str(X_current_plus_prior.shape), str(prior_pred_all.shape)],
}))

print("Global train medians:", global_train_medians)
print("Historical prior feature columns:", HIST_FEATURE_COLUMNS)

,matrix,shape
0,X_current,"(73972, 64)"
1,X_current_plus_prior,"(73972, 72)"
2,prior_pred_all,"(73972, 3)"


Global train medians: {'truck_q25': 1494.1099999999997, 'truck_q50': 2314.45, 'truck_q75': 3631.97}
Historical prior feature columns: ['hist_truck_q25', 'hist_truck_q50', 'hist_truck_q75', 'hist_truck_iqr', 'hist_truck_q75_q50_gap', 'hist_truck_q50_q25_gap', 'hist_n_prior_years', 'hist_has_prior']


## 12. Register reference baselines

These two rows are included so that the M0.5 leaderboard is interpretable without opening the previous M0 output.

In [14]:
# -----------------------------------------------------------------------------
# Register reference baselines
# -----------------------------------------------------------------------------
register_model_predictions(
    model="global_train_median",
    raw_pred_all=global_pred_all,
    df=df_model,
    masks=masks,
    y_all=y_all,
    weights_all=weights_all,
    orig_col=orig_col,
    dest_col=dest_col,
    year_col=year_col,
    variant="postprocessed",
    extra_note={"description": "Global train median for q25/q50/q75."},
)

register_model_predictions(
    model="od_historical_prior",
    raw_pred_all=prior_pred_all,
    df=df_model,
    masks=masks,
    y_all=y_all,
    weights_all=weights_all,
    orig_col=orig_col,
    dest_col=dest_col,
    year_col=year_col,
    variant="postprocessed",
    extra_note={
        "description": "Train-only OD historical median. Train rows use leave-one-training-row-out priors.",
        "global_train_medians_for_cold_od": global_train_medians,
    },
)

print("Registered reference baselines.")

Registered reference baselines.


## 13. Direct LightGBM quantile baseline

This re-trains the direct M0 LightGBM quantile model inside the M0.5 notebook. It is needed for ensemble and residual comparisons.

In [15]:
# -----------------------------------------------------------------------------
# Direct LightGBM quantile model: current features -> q25/q50/q75
# -----------------------------------------------------------------------------
if config.run_direct_lightgbm:
    direct_lgbm_raw_all, direct_lgbm_models = fit_quantile_boosting_models(
        X_train=X_current_split["train"],
        y_train=y_split["train"],
        X_val=X_current_split["val"],
        y_val=y_split["val"],
        X_all=X_current,
        weights_train=w_split["train"],
        weights_val=w_split["val"],
        config=config,
        model_name="lightgbm_direct",
    )

    direct_lgbm_pred_all = register_model_predictions(
        model="lightgbm_direct",
        raw_pred_all=direct_lgbm_raw_all,
        df=df_model,
        masks=masks,
        y_all=y_all,
        weights_all=weights_all,
        orig_col=orig_col,
        dest_col=dest_col,
        year_col=year_col,
        variant="postprocessed",
        extra_note={"features": feature_columns},
    )

    if config.save_models:
        joblib.dump(
            {
                "models": direct_lgbm_models,
                "feature_columns": feature_columns,
                "imputer": imputer_current,
                "model_type": "direct_lightgbm_quantile",
            },
            models_dir / "lightgbm_direct.joblib",
        )
else:
    direct_lgbm_raw_all = None
    direct_lgbm_pred_all = None
    direct_lgbm_models = []
    print("Skipped direct LightGBM by configuration.")

Training lightgbm_direct: quantile tau=0.25
Training lightgbm_direct: quantile tau=0.50
Training lightgbm_direct: quantile tau=0.75


## 14. Validation-tuned OD-history + LightGBM ensembles

These are the simplest M0.5 hybrids. They do not train a new residual model; they only combine two already-trained predictors using the validation year.

In [16]:
# -----------------------------------------------------------------------------
# Convex ensembles of OD historical prior and direct LightGBM
# -----------------------------------------------------------------------------
ensemble_weight_tables = []

if config.run_ensembles and direct_lgbm_raw_all is not None:
    # Global alpha shared by q25/q50/q75.
    best_alpha, global_weight_table = tune_global_convex_weight(
        y_val=y_split["val"],
        pred_a_val=prior_pred_all[masks["val"]],
        pred_b_val=direct_lgbm_pred_all[masks["val"]],
        weights_val=w_split["val"],
        grid_step=config.ensemble_grid_step,
    )
    global_weight_table.insert(0, "ensemble", "hist_lgbm_ensemble_global")
    ensemble_weight_tables.append(global_weight_table)

    ensemble_global_raw_all = best_alpha * prior_pred_all + (1.0 - best_alpha) * direct_lgbm_pred_all
    register_model_predictions(
        model="hist_lgbm_ensemble_global",
        raw_pred_all=ensemble_global_raw_all,
        df=df_model,
        masks=masks,
        y_all=y_all,
        weights_all=weights_all,
        orig_col=orig_col,
        dest_col=dest_col,
        year_col=year_col,
        variant="postprocessed",
        extra_note={"alpha_history": best_alpha, "alpha_lightgbm": 1.0 - best_alpha},
    )
    print(f"Best global ensemble alpha_history = {best_alpha:.4f}")

    # One alpha per quantile.
    best_alphas, per_q_weight_table = tune_per_quantile_convex_weights(
        y_val=y_split["val"],
        pred_a_val=prior_pred_all[masks["val"]],
        pred_b_val=direct_lgbm_pred_all[masks["val"]],
        weights_val=w_split["val"],
        grid_step=config.ensemble_grid_step,
    )
    per_q_weight_table.insert(0, "ensemble", "hist_lgbm_ensemble_per_quantile")
    ensemble_weight_tables.append(per_q_weight_table)

    ensemble_per_q_raw_all = apply_per_quantile_blend(prior_pred_all, direct_lgbm_pred_all, best_alphas)
    register_model_predictions(
        model="hist_lgbm_ensemble_per_quantile",
        raw_pred_all=ensemble_per_q_raw_all,
        df=df_model,
        masks=masks,
        y_all=y_all,
        weights_all=weights_all,
        orig_col=orig_col,
        dest_col=dest_col,
        year_col=year_col,
        variant="postprocessed",
        extra_note={
            "alpha_history_q25": float(best_alphas[0]),
            "alpha_history_q50": float(best_alphas[1]),
            "alpha_history_q75": float(best_alphas[2]),
            "alpha_lightgbm_q25": float(1.0 - best_alphas[0]),
            "alpha_lightgbm_q50": float(1.0 - best_alphas[1]),
            "alpha_lightgbm_q75": float(1.0 - best_alphas[2]),
        },
    )
    print("Best per-quantile alpha_history:", dict(zip(LABEL_COLUMNS, best_alphas)))
else:
    print("Skipped ensembles because direct LightGBM was skipped or unavailable.")

Best global ensemble alpha_history = 0.8400
Best per-quantile alpha_history: {'truck_q25': np.float64(0.6), 'truck_q50': np.float64(1.0), 'truck_q75': np.float64(0.88)}


## 15. Residual LightGBM using current features

This is the core M0.5 idea:

```text
residual_qτ = truck_qτ - historical_prior_qτ
final_prediction_qτ = historical_prior_qτ + LightGBM_residual_qτ
```

The model below uses only the original manifest features to predict the residual.

In [17]:
# -----------------------------------------------------------------------------
# Residual LightGBM with current manifest features only
# -----------------------------------------------------------------------------
if config.run_residual_current_features:
    residual_target_all = y_all - prior_pred_all
    residual_split = labels_by_split(residual_target_all, masks)

    residual_current_raw_resid_all, residual_current_models = fit_quantile_boosting_models(
        X_train=X_current_split["train"],
        y_train=residual_split["train"],
        X_val=X_current_split["val"],
        y_val=residual_split["val"],
        X_all=X_current,
        weights_train=w_split["train"],
        weights_val=w_split["val"],
        config=config,
        model_name="residual_lgbm_current_features",
    )

    residual_current_raw_all = prior_pred_all + residual_current_raw_resid_all
    residual_current_pred_all = register_model_predictions(
        model="residual_lgbm_current_features",
        raw_pred_all=residual_current_raw_all,
        df=df_model,
        masks=masks,
        y_all=y_all,
        weights_all=weights_all,
        orig_col=orig_col,
        dest_col=dest_col,
        year_col=year_col,
        variant="postprocessed",
        extra_note={
            "base_model": "od_historical_prior",
            "residual_features": feature_columns,
        },
    )

    if config.save_models:
        joblib.dump(
            {
                "models": residual_current_models,
                "feature_columns": feature_columns,
                "imputer": imputer_current,
                "model_type": "residual_lightgbm_current_features",
            },
            models_dir / "residual_lgbm_current_features.joblib",
        )
else:
    residual_current_raw_all = None
    residual_current_pred_all = None
    residual_current_models = []
    print("Skipped residual LightGBM with current features by configuration.")

Training residual_lgbm_current_features: quantile tau=0.25
Training residual_lgbm_current_features: quantile tau=0.50
Training residual_lgbm_current_features: quantile tau=0.75


## 16. Residual LightGBM using current features plus historical-prior features

This is usually the strongest M0.5 candidate because it lets the residual model condition on:

- demand / mode / zone features, and
- historical q25/q50/q75 level and interval width.

The historical-prior features are still train-only and therefore do not leak validation or test labels.

In [18]:
# -----------------------------------------------------------------------------
# Residual LightGBM with current features plus historical-prior features
# -----------------------------------------------------------------------------
if config.run_residual_prior_features:
    residual_target_all = y_all - prior_pred_all
    residual_split = labels_by_split(residual_target_all, masks)

    residual_prior_raw_resid_all, residual_prior_models = fit_quantile_boosting_models(
        X_train=X_prior_split["train"],
        y_train=residual_split["train"],
        X_val=X_prior_split["val"],
        y_val=residual_split["val"],
        X_all=X_current_plus_prior,
        weights_train=w_split["train"],
        weights_val=w_split["val"],
        config=config,
        model_name="residual_lgbm_prior_features",
    )

    residual_prior_raw_all = prior_pred_all + residual_prior_raw_resid_all
    residual_prior_pred_all = register_model_predictions(
        model="residual_lgbm_prior_features",
        raw_pred_all=residual_prior_raw_all,
        df=df_model,
        masks=masks,
        y_all=y_all,
        weights_all=weights_all,
        orig_col=orig_col,
        dest_col=dest_col,
        year_col=year_col,
        variant="postprocessed",
        extra_note={
            "base_model": "od_historical_prior",
            "residual_features": feature_columns_with_prior,
        },
    )

    if config.save_models:
        joblib.dump(
            {
                "models": residual_prior_models,
                "feature_columns": feature_columns_with_prior,
                "imputer": imputer_current_plus_prior,
                "model_type": "residual_lightgbm_prior_features",
            },
            models_dir / "residual_lgbm_prior_features.joblib",
        )
else:
    residual_prior_raw_all = None
    residual_prior_pred_all = None
    residual_prior_models = []
    print("Skipped residual LightGBM with prior features by configuration.")

Training residual_lgbm_prior_features: quantile tau=0.25
Training residual_lgbm_prior_features: quantile tau=0.50
Training residual_lgbm_prior_features: quantile tau=0.75


## 17. Direct LightGBM with historical-prior features

This model predicts q25/q50/q75 directly from:

```text
original manifest features + train-only historical-prior features
```

It tests whether residual learning is necessary or whether direct quantile learning with prior features is enough.

In [19]:
# -----------------------------------------------------------------------------
# Direct LightGBM with current features plus historical-prior features
# -----------------------------------------------------------------------------
if config.run_prior_feature_direct_lightgbm:
    prior_feature_direct_raw_all, prior_feature_direct_models = fit_quantile_boosting_models(
        X_train=X_prior_split["train"],
        y_train=y_split["train"],
        X_val=X_prior_split["val"],
        y_val=y_split["val"],
        X_all=X_current_plus_prior,
        weights_train=w_split["train"],
        weights_val=w_split["val"],
        config=config,
        model_name="prior_feature_lgbm_direct",
    )

    prior_feature_direct_pred_all = register_model_predictions(
        model="prior_feature_lgbm_direct",
        raw_pred_all=prior_feature_direct_raw_all,
        df=df_model,
        masks=masks,
        y_all=y_all,
        weights_all=weights_all,
        orig_col=orig_col,
        dest_col=dest_col,
        year_col=year_col,
        variant="postprocessed",
        extra_note={"features": feature_columns_with_prior},
    )

    if config.save_models:
        joblib.dump(
            {
                "models": prior_feature_direct_models,
                "feature_columns": feature_columns_with_prior,
                "imputer": imputer_current_plus_prior,
                "model_type": "prior_feature_lgbm_direct",
            },
            models_dir / "prior_feature_lgbm_direct.joblib",
        )
else:
    prior_feature_direct_raw_all = None
    prior_feature_direct_pred_all = None
    prior_feature_direct_models = []
    print("Skipped direct LightGBM with prior features by configuration.")

Training prior_feature_lgbm_direct: quantile tau=0.25
Training prior_feature_lgbm_direct: quantile tau=0.50
Training prior_feature_lgbm_direct: quantile tau=0.75


## 18. Save outputs

This cell writes all experiment artifacts.

Expected files:

```text
metrics.csv
leaderboard_test.csv
predictions_val_test.parquet or predictions_val_test.csv
ensemble_weights.csv
run_config.json
feature_columns_used.json
model_notes.json
models/*.joblib
```

In [20]:
# -----------------------------------------------------------------------------
# Save metrics, predictions, config, and notes
# -----------------------------------------------------------------------------
metrics_df = pd.DataFrame(metrics_rows)

# Stable column order: identifiers first, then metrics.
front_cols = ["model", "variant", "split", "n_rows"]
other_cols = [c for c in metrics_df.columns if c not in front_cols]
metrics_df = metrics_df[front_cols + other_cols]

metrics_path = output_dir / "metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

leaderboard_test = (
    metrics_df.loc[metrics_df["split"] == "test"]
    .sort_values("pinball_mean", ascending=True)
    .reset_index(drop=True)
)
leaderboard_test.insert(0, "rank_by_test_pinball", np.arange(1, len(leaderboard_test) + 1))
leaderboard_path = output_dir / "leaderboard_test.csv"
leaderboard_test.to_csv(leaderboard_path, index=False)

predictions_df = pd.concat(prediction_tables, ignore_index=True) if prediction_tables else pd.DataFrame()
predictions_path = safe_write_table(predictions_df, output_dir / "predictions_val_test")

if ensemble_weight_tables:
    ensemble_weights_df = pd.concat(ensemble_weight_tables, ignore_index=True)
else:
    ensemble_weights_df = pd.DataFrame()
ensemble_weights_path = output_dir / "ensemble_weights.csv"
ensemble_weights_df.to_csv(ensemble_weights_path, index=False)

feature_info = {
    "manifest_feature_columns": feature_columns,
    "historical_prior_feature_columns": HIST_FEATURE_COLUMNS,
    "feature_columns_with_prior": feature_columns_with_prior,
    "label_columns": LABEL_COLUMNS,
}
feature_path = output_dir / "feature_columns_used.json"
feature_path.write_text(json.dumps(json_safe(feature_info), indent=2), encoding="utf-8")

run_config = {
    "notebook": "FREIGHT-MNet M0.5 Hybrid Baseline Notebook",
    "created_at_unix": time.time(),
    "config": json_safe(asdict(config)),
    "paths": json_safe(paths),
    "dataset": {
        "n_rows": int(len(df)),
        "n_features_manifest": int(len(feature_columns)),
        "n_features_with_prior": int(len(feature_columns_with_prior)),
        "label_columns": LABEL_COLUMNS,
        "split_counts": {k: int(v.sum()) for k, v in masks.items()},
        "year_counts": {str(k): int(v) for k, v in df[year_col].value_counts().sort_index().to_dict().items()},
        "origin_column": orig_col,
        "destination_column": dest_col,
        "year_column": year_col,
    },
    "package_versions": {
        "python": sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "lightgbm_available": LIGHTGBM_AVAILABLE,
        "lightgbm": getattr(lgb, "__version__", None) if LIGHTGBM_AVAILABLE else None,
    },
}
run_config_path = output_dir / "run_config.json"
run_config_path.write_text(json.dumps(json_safe(run_config), indent=2), encoding="utf-8")

model_notes_path = output_dir / "model_notes.json"
model_notes_path.write_text(json.dumps(json_safe(model_notes), indent=2), encoding="utf-8")

print("Saved outputs:")
for p in [metrics_path, leaderboard_path, predictions_path, ensemble_weights_path, feature_path, run_config_path, model_notes_path]:
    print(" -", p)

Saved outputs:
 - E:\NetworkOptimization\pythonProject1\Data\10_experiments\m05_hybrid_baselines_v1_notebook\east_plus_gulf\metrics.csv
 - E:\NetworkOptimization\pythonProject1\Data\10_experiments\m05_hybrid_baselines_v1_notebook\east_plus_gulf\leaderboard_test.csv
 - E:\NetworkOptimization\pythonProject1\Data\10_experiments\m05_hybrid_baselines_v1_notebook\east_plus_gulf\predictions_val_test.parquet
 - E:\NetworkOptimization\pythonProject1\Data\10_experiments\m05_hybrid_baselines_v1_notebook\east_plus_gulf\ensemble_weights.csv
 - E:\NetworkOptimization\pythonProject1\Data\10_experiments\m05_hybrid_baselines_v1_notebook\east_plus_gulf\feature_columns_used.json
 - E:\NetworkOptimization\pythonProject1\Data\10_experiments\m05_hybrid_baselines_v1_notebook\east_plus_gulf\run_config.json
 - E:\NetworkOptimization\pythonProject1\Data\10_experiments\m05_hybrid_baselines_v1_notebook\east_plus_gulf\model_notes.json


## 19. Inspect leaderboard

The best model should be judged primarily on `test pinball_mean`, but also check:

- `mae_q75`
- `iqr_mae`
- `stress_top10_mae_q75`
- `sparse_bottom25_mae_q75`
- `raw_crossing_rate`

In [21]:
# -----------------------------------------------------------------------------
# Display compact test leaderboard
# -----------------------------------------------------------------------------
important_cols = [
    "rank_by_test_pinball",
    "model",
    "variant",
    "n_rows",
    "pinball_mean",
    "mae_q50",
    "mae_q75",
    "iqr_mae",
    "stress_top10_mae_q75",
    "sparse_bottom25_mae_q75",
    "raw_crossing_rate",
]
compact_leaderboard = leaderboard_test[[c for c in important_cols if c in leaderboard_test.columns]].copy()
display(compact_leaderboard)

,rank_by_test_pinball,model,variant,n_rows,pinball_mean,mae_q50,mae_q75,iqr_mae,stress_top10_mae_q75,sparse_bottom25_mae_q75,raw_crossing_rate
0,1,residual_lgbm_prior_features,postprocessed,10574,131.187824,230.953405,451.623436,384.731525,607.540468,557.575446,0.000095
1,2,prior_feature_lgbm_direct,postprocessed,10574,131.304712,231.615675,449.055148,381.889820,605.877011,554.195459,0.000000
2,3,residual_lgbm_current_features,postprocessed,10574,132.558496,230.401410,465.240711,395.618681,634.736172,573.990126,0.000095
3,4,hist_lgbm_ensemble_global,postprocessed,10574,146.728685,231.257504,462.529352,370.897768,694.091366,562.211623,0.000000
4,5,hist_lgbm_ensemble_per_quantile,postprocessed,10574,148.175706,230.515979,463.454406,370.358502,697.644912,563.760143,0.000284
5,6,od_historical_prior,postprocessed,10574,153.000149,230.534636,472.672178,385.553897,709.868526,577.776247,0.000000
6,7,lightgbm_direct,postprocessed,10574,157.521557,283.130157,571.960199,521.059845,690.011190,684.617810,0.001135
7,8,global_train_median,postprocessed,10574,553.457953,989.927425,1500.794747,843.408850,3312.300293,1705.145227,0.000000


## 20. Compare against previous M0 output if available

This optional diagnostic looks for the previous M0 notebook's output folder and prints a side-by-side leaderboard. It does not affect the M0.5 model fitting.

In [22]:
# -----------------------------------------------------------------------------
# Optional comparison with previous M0 baseline output
# -----------------------------------------------------------------------------
previous_m0_dir = config.data_root / "10_experiments" / "m0_baselines_v1_notebook" / scope
previous_m0_leaderboard = previous_m0_dir / "leaderboard_test.csv"

if previous_m0_leaderboard.exists():
    m0_lb = pd.read_csv(previous_m0_leaderboard)
    m0_lb = m0_lb.copy()
    m0_lb.insert(0, "experiment", "M0")
    m05_lb = leaderboard_test.copy()
    m05_lb.insert(0, "experiment", "M0.5")
    common_cols = [
        "experiment", "model", "variant", "pinball_mean", "mae_q50", "mae_q75",
        "iqr_mae", "stress_top10_mae_q75", "sparse_bottom25_mae_q75", "raw_crossing_rate"
    ]
    comparison = pd.concat([
        m0_lb[[c for c in common_cols if c in m0_lb.columns]],
        m05_lb[[c for c in common_cols if c in m05_lb.columns]],
    ], ignore_index=True)
    comparison = comparison.sort_values("pinball_mean", ascending=True).reset_index(drop=True)
    display(comparison.head(20))
else:
    print("Previous M0 leaderboard not found:", previous_m0_leaderboard)

,experiment,model,variant,pinball_mean,mae_q50,mae_q75,iqr_mae,stress_top10_mae_q75,sparse_bottom25_mae_q75,raw_crossing_rate
0,M0.5,residual_lgbm_prior_features,postprocessed,131.187824,230.953405,451.623436,384.731525,607.540468,557.575446,0.000095
1,M0.5,prior_feature_lgbm_direct,postprocessed,131.304712,231.615675,449.055148,381.889820,605.877011,554.195459,0.000000
2,M0.5,residual_lgbm_current_features,postprocessed,132.558496,230.401410,465.240711,395.618681,634.736172,573.990126,0.000095
3,M0.5,hist_lgbm_ensemble_global,postprocessed,146.728685,231.257504,462.529352,370.897768,694.091366,562.211623,0.000000
4,M0.5,hist_lgbm_ensemble_per_quantile,postprocessed,148.175706,230.515979,463.454406,370.358502,697.644912,563.760143,0.000284
5,M0,od_historical_median,postprocessed,153.000149,230.534636,472.672178,385.553897,709.868526,577.776247,0.000000
6,M0.5,od_historical_prior,postprocessed,153.000149,230.534636,472.672178,385.553897,709.868526,577.776247,0.000000
7,M0.5,lightgbm_direct,postprocessed,157.521557,283.130157,571.960199,521.059845,690.011190,684.617810,0.001135
8,M0,lightgbm_quantile,postprocessed,157.827689,281.569916,575.876020,524.510670,687.376490,690.101260,0.000757
9,M0,od_historical_median,val_median_calibrated,158.227579,244.008809,482.032129,386.411862,724.578299,587.356420,0.000000
